# 🎬 CogVideoX-5B-I2V (Google Colab - RAM Segura)
Este notebook está otimizado para rodar no Colab sem estourar a RAM do sistema, mesmo com modelos grandes.
- Usa `device_map="balanced"` e `offload_folder`
- Gera vídeos a partir de imagens com até 16 frames
- Integração com Hugging Face para acesso ao modelo

In [ ]:
# ✅ Instalar dependências
!pip install diffusers==0.33.1 transformers accelerate einops gradio ffmpeg-python safetensors --quiet

In [ ]:
# 🔑 Login na Hugging Face (necessário para baixar o modelo CogVideoX)
from huggingface_hub import login
login()

In [ ]:
# 📦 Imports e configurações gerais
import os
import torch
from PIL import Image
import gradio as gr
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs("outputs", exist_ok=True)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# 🚀 Carregar componentes separadamente com controle de memória
from transformers import logging
logging.set_verbosity_error()
from diffusers import CogVideoXTransformer3DModel, AutoencoderKLCogVideoX

# Carregar transformer manualmente (com offload de pesos para disco)
transformer = CogVideoXTransformer3DModel.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="transformer",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)

# Carregar o VAE manualmente
vae = AutoencoderKLCogVideoX.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    subfolder="vae",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)

# Agora sim: carregar a pipeline passando apenas o que é compatível
pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    transformer=transformer,
    vae=vae,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="balanced",
    offload_folder="modelo_offload"
)

In [ ]:
# 🎥 Geração com parâmetros configuráveis
def generate_video(image_path, prompt, width, height, frames, steps):
    image = load_image(image_path).convert("RGB").resize((width, height))
    result = pipe(
        image=image,
        prompt=prompt,
        guidance_scale=5,
        num_inference_steps=steps,
        num_frames=frames
    )
    frames = result.frames[0]
    out = f"outputs/cogvideo_{frames[0].size[0]}x{frames[0].size[1]}.mp4"
    export_to_video(frames, out, fps=6)
    return out

In [ ]:
# 🖼️ Interface Gradio com controles
with gr.Blocks() as demo:
    gr.Markdown("## CogVideoX - Geração de vídeo a partir de imagem 🎬")
    with gr.Row():
        img = gr.Image(type="filepath", label="Imagem")
        prm = gr.Textbox(label="Prompt (ex: 'neve caindo à noite')")
    with gr.Row():
        width = gr.Slider(256, 1280, value=720, step=64, label="Largura")
        height = gr.Slider(256, 720, value=480, step=64, label="Altura")
    with gr.Row():
        frames = gr.Slider(2, 16, value=8, step=1, label="Nº de Frames")
        steps = gr.Slider(4, 50, value=16, step=1, label="Inference Steps")
    btn = gr.Button("🎬 Gerar Vídeo")
    vid = gr.Video(label="🎞️ Resultado")
    btn.click(fn=generate_video, inputs=[img, prm, width, height, frames, steps], outputs=vid)
    demo.launch(share=True)